# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [6]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, precision_recall_curve,
                             roc_auc_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
import matplotlib.pyplot as plt

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

## 1. Inspect target distribution to understand imbalance.

In [4]:
# Basic shape and target distribution
fraud.shape, fraud['fraud'].value_counts(), fraud['fraud'].value_counts(normalize=True).round(4)

((1000000, 8),
 fraud
 0.0    912597
 1.0     87403
 Name: count, dtype: int64,
 fraud
 0.0    0.9126
 1.0    0.0874
 Name: proportion, dtype: float64)

**Comment:** If the percentage of `fraud=1` is much smaller than `fraud=0` (e.g., a few percent), then the dataset is **imbalanced**. Accuracy alone can be misleading; we will use **PR-AUC** and per-class metrics.

**2. Train/Test split + preprocessing (scaling numeric features)**
- We split once and keep the **test set** untouched for all experiments.
- We scale only **numeric** features; binary 0/1 columns pass through unchanged.
- Scaling helps Logistic Regression converge and keeps coefficients comparable.

In [5]:
# Identify columns by type (based on provided metadata)
target = 'fraud'
num_cols = ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price']
bin_cols = ['repeat_retailer', 'used_chip', 'used_pin_number', 'online_order']

X = fraud[num_cols + bin_cols].copy()
y = fraud[target].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4)

((800000, 7), (200000, 7), np.float64(0.0874), np.float64(0.0874))

In [7]:
# Preprocessing: scale numeric features only; passthrough the binary features
def evaluate(y_true, y_proba, y_pred, title="Model"):
    """Print key metrics for imbalanced classification.
    y_proba: predicted probabilities for the positive class (fraud=1)
    y_pred:  class predictions at the default 0.5 threshold
    """
    ap = average_precision_score(y_true, y_proba)  # PR-AUC (Average Precision)
    roc = roc_auc_score(y_true, y_proba)
    print(f"=== {title} ===")
    print(f"PR-AUC (Average Precision): {ap:.4f}")
    print(f"ROC-AUC: {roc:.4f}")
    print("\nConfusion matrix (threshold=0.5):\n", confusion_matrix(y_true, y_pred))
    print("\nClassification report (threshold=0.5):\n", classification_report(y_true, y_pred, digits=4))
    return ap, roc

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
    ],
    remainder='passthrough'  # keep binary columns as they are
)

# Baseline Logistic Regression pipeline (no resampling)
base_pipe = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
base_pipe.fit(X_train, y_train)
y_proba_base = base_pipe.predict_proba(X_test)[:, 1]
y_pred_base = (y_proba_base >= 0.5).astype(int)
ap_base, roc_base = evaluate(y_test, y_proba_base, y_pred_base, title='Baseline LogisticRegression')

=== Baseline LogisticRegression ===
PR-AUC (Average Precision): 0.8068
ROC-AUC: 0.9671

Confusion matrix (threshold=0.5):
 [[181293   1226]
 [  6904  10577]]

Classification report (threshold=0.5):
               precision    recall  f1-score   support

           0     0.9633    0.9933    0.9781    182519
           1     0.8961    0.6051    0.7224     17481

    accuracy                         0.9594    200000
   macro avg     0.9297    0.7992    0.8502    200000
weighted avg     0.9574    0.9594    0.9557    200000



**Interpretation:**
- Focus on **PR-AUC** and the **recall** for the fraud class (1). High recall means we catch more frauds.
- If PR-AUC is low and recall for class 1 is poor, the model is likely biased toward the majority class (legit=0).

**4. Oversampling (RandomOverSampler)**


In [8]:
over_pipe = ImbPipeline([
    ('prep', preprocess),
    ('over', RandomOverSampler(random_state=42)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
over_pipe.fit(X_train, y_train)
y_proba_over = over_pipe.predict_proba(X_test)[:, 1]
y_pred_over = (y_proba_over >= 0.5).astype(int)
ap_over, roc_over = evaluate(y_test, y_proba_over, y_pred_over, title='LogReg + Oversample (ROS)')

=== LogReg + Oversample (ROS) ===
PR-AUC (Average Precision): 0.7573
ROC-AUC: 0.9795

Confusion matrix (threshold=0.5):
 [[170388  12131]
 [   908  16573]]

Classification report (threshold=0.5):
               precision    recall  f1-score   support

           0     0.9947    0.9335    0.9631    182519
           1     0.5774    0.9481    0.7177     17481

    accuracy                         0.9348    200000
   macro avg     0.7860    0.9408    0.8404    200000
weighted avg     0.9582    0.9348    0.9417    200000



**Comment:** Compare PR-AUC and the recall for class 1 with the baseline. Oversampling often **improves recall** for the minority (fraud) at the cost of some precision.

## 5. Undersampling (RandomUnderSampler) on **train only**
We undersample the majority class in the **training set** so the model sees a balanced dataset. This reduces training size and can increase variance but may improve recall.

In [9]:
under_pipe = ImbPipeline([
    ('prep', preprocess),
    ('under', RandomUnderSampler(random_state=42)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
under_pipe.fit(X_train, y_train)
y_proba_under = under_pipe.predict_proba(X_test)[:, 1]
y_pred_under = (y_proba_under >= 0.5).astype(int)
ap_under, roc_under = evaluate(y_test, y_proba_under, y_pred_under, title='LogReg + Undersample (RUS)')

=== LogReg + Undersample (RUS) ===
PR-AUC (Average Precision): 0.7564
ROC-AUC: 0.9796

Confusion matrix (threshold=0.5):
 [[170380  12139]
 [   917  16564]]

Classification report (threshold=0.5):
               precision    recall  f1-score   support

           0     0.9946    0.9335    0.9631    182519
           1     0.5771    0.9475    0.7173     17481

    accuracy                         0.9347    200000
   macro avg     0.7859    0.9405    0.8402    200000
weighted avg     0.9581    0.9347    0.9416    200000



**Comment:** Undersampling may **boost recall** but sometimes lowers precision and overall stability because we learn from fewer majority examples.

## 6. SMOTE (Synthetic Minority Over-sampling Technique)
SMOTE synthesizes new minority samples in feature space, balancing the classes while keeping more information than pure undersampling.

In [10]:
smote_pipe = ImbPipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42, k_neighbors=5)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
smote_pipe.fit(X_train, y_train)
y_proba_smote = smote_pipe.predict_proba(X_test)[:, 1]
y_pred_smote = (y_proba_smote >= 0.5).astype(int)
ap_smote, roc_smote = evaluate(y_test, y_proba_smote, y_pred_smote, title='LogReg + SMOTE')

=== LogReg + SMOTE ===
PR-AUC (Average Precision): 0.7574
ROC-AUC: 0.9795

Confusion matrix (threshold=0.5):
 [[170384  12135]
 [   909  16572]]

Classification report (threshold=0.5):
               precision    recall  f1-score   support

           0     0.9947    0.9335    0.9631    182519
           1     0.5773    0.9480    0.7176     17481

    accuracy                         0.9348    200000
   macro avg     0.7860    0.9408    0.8404    200000
weighted avg     0.9582    0.9348    0.9417    200000



**Comment:** SMOTE often gives a good balance between recall and precision compared with naive oversampling/undersampling.

## Summary comparison (PR-AUC & ROC-AUC)
We compare the key metrics across all four setups.

In [11]:
summary = pd.DataFrame({
    'Model': ['Baseline', 'Oversample (ROS)', 'Undersample (RUS)', 'SMOTE'],
    'PR-AUC': [ap_base, ap_over, ap_under, ap_smote],
    'ROC-AUC': [roc_base, roc_over, roc_under, roc_smote]
}).sort_values('PR-AUC', ascending=False).reset_index(drop=True)
summary

,Model,PR-AUC,ROC-AUC
0,Baseline,0.806772,0.967053
1,SMOTE,0.757411,0.979523
2,Oversample (ROS),0.757259,0.979535
3,Undersample (RUS),0.756384,0.979583


**Answers to Does it improve the performance of our model?**

**Oversample: No.** PR-AUC dropped from 0.8068 → 0.7573. (ROC-AUC is a bit higher, but for fraud—an imbalanced task—PR-AUC is the right yardstick.)

**Undersample: No.** PR-AUC dropped to 0.7564.

**SMOTE: No.** PR-AUC dropped to 0.7574.